In [1]:
from pyspark.sql import SparkSession

# Local Spark session with required JARs
spark = SparkSession.builder \
    .appName("test_connection") \
    .master("local[*]") \
    .config("spark.jars", "/jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/postgresql-42.7.2.jar") \
    .getOrCreate()

print(f"✅ Spark Version: {spark.version}")  # 3.5.0

# Test Doris connector classpath
try:
    spark.read.format("doris").option("doris.fenodes", "test:8030")
    print("✅ Doris connector is available in classpath")
except Exception as e:
    if "ClassNotFoundException" in str(e):
        print("❌ Doris connector NOT found")
    else:
        print(f"⚠️ Expected connection error: {type(e).__name__}")

✅ Spark Version: 3.5.0
✅ Doris connector is available in classpath


In [2]:
from pyspark.sql import SparkSession

# Build session that connects to EXTERNAL cluster
spark = SparkSession.builder \
    .appName("verify_cluster") \
    .master("spark://spark-master:7077") \
    .config("spark.jars", "/jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/postgresql-42.7.2.jar") \
    .config("spark.driver.host", "jupyter-lab") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .getOrCreate()

# Diagnostic output
print(f"🔗 Driver Spark version: {spark.version}")  # Client: 3.5.0
print(f"🔗 Cluster Spark version: {spark.sparkContext.version}")  # Cluster: 3.5.8
print(f"🔗 Master URL: {spark.sparkContext.master}")  # Should be: spark://spark-master:7077

# Test distributed execution: force a job that runs on workers
print("\n🧪 Testing distributed execution...")
data = [(i, f"user_{i}") for i in range(1000)]
df = spark.createDataFrame(data, ["id", "name"])

# This transformation triggers execution on cluster workers
result = df.rdd.map(lambda row: (row.id * 2, row.name.upper())).collect()

print(f"✅ Processed {len(result)} rows on cluster workers")
print(f"✅ Sample output: {result[:3]}")

# Check if Doris connector is available on executors
try:
    spark.read.format("doris").option("doris.fenodes", "test:8030").load()
except Exception as e:
    if "ClassNotFoundException" in str(e):
        print("❌ Doris connector NOT available on executors")
    else:
        print(f"✅ Connector loaded (connection error expected): {type(e).__name__}")

spark.stop()

🔗 Driver Spark version: 3.5.0
🔗 Cluster Spark version: 3.5.0
🔗 Master URL: local[*]

🧪 Testing distributed execution...
✅ Processed 1000 rows on cluster workers
✅ Sample output: [(0, 'USER_0'), (2, 'USER_1'), (4, 'USER_2')]
✅ Connector loaded (connection error expected): Py4JJavaError


In [ ]:
from pyspark.sql import SparkSession
import os

# 🔑 CRITICAL: Build session with EXTERNAL cluster connection
spark = SparkSession.builder \
    .appName("verify_cluster_connection") \
    .master("spark://spark-master:7077") \
    .config("spark.jars", "/jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/postgresql-42.7.2.jar") \
    .config("spark.driver.host", "jupyter-lab") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.network.timeout", "800s") \
    .getOrCreate()

# Diagnostic output
print(f"🔗 Driver Spark version: {spark.version}")           # Client: 3.5.0 (expected)
print(f"🔗 Cluster Spark version: {spark.sparkContext.version}")  # Cluster: should be 3.5.8 ✅
print(f"🔗 Master URL: {spark.sparkContext.master}")         # Should be: spark://spark-master:7077 ✅

# Test distributed execution
print("\n🧪 Testing distributed execution on cluster...")
data = [(i, f"user_{i}") for i in range(1000)]
df = spark.createDataFrame(data, ["id", "name"])

# This forces execution on cluster executors
result = df.rdd.map(lambda row: (row.id * 2, row.name.upper())).collect()

print(f"✅ Processed {len(result)} rows on cluster workers")
print(f"✅ Sample: {result[:3]}")

# Verify Doris connector availability
try:
    spark.read.format("doris").option("doris.fenodes", "test:8030").load()
except Exception as e:
    if "ClassNotFoundException" in str(e):
        print("❌ Doris connector NOT in executor classpath")
    else:
        print(f"✅ Connector loaded (expected connection error): {type(e).__name__}")

spark.stop()

In [5]:
# In Jupyter notebook - validation cell
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("post_fix_validation") \
    .master("spark://spark-master:7077") \
    .config("spark.jars", "/jars/spark-doris-connector-spark-3.5-25.2.0.jar,/jars/postgresql-42.7.2.jar") \
    .config("spark.driver.host", "jupyter-lab") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .getOrCreate()

print(f"✅ Driver: {spark.version}")
print(f"✅ Cluster: {spark.sparkContext.version}")
print(f"✅ Master: {spark.sparkContext.master}")

# Quick test
result = spark.range(10).collect()
print(f"✅ Test: {len(result)} rows collected")

spark.stop()

✅ Driver: 3.5.0
✅ Cluster: 3.5.0
✅ Master: spark://spark-master:7077
✅ Test: 10 rows collected
